# Lesson 4.4: Sample Proportions

Simulate thousands of polls to see that the spread of the sample proportion matches the formula square root of p(1-p)/n.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A polling agency asks 1,500 voters whether they support an environmental bill, and the headline says 54% with a margin of error of 2.5 points. Where does the 2.5 come from? We pretend the true support is exactly 0.54, run polls on the computer, and see how much the result varies.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: One poll

The lesson's poll: the true support for a bill is p = 0.54 and we ask n = 1,500 voters. Each voter says yes with probability p, so the number of yes answers is a binomial draw.

In [1]:
import numpy as np  # Load NumPy for random numbers and arithmetic.

rng = np.random.default_rng(2024)  # Create a seeded random generator (seed 2024) so results are repeatable.
p, n = 0.54, 1500  # Set the true support p = 0.54 and the number of voters polled n = 1,500.

yes = rng.binomial(n, p)  # Simulate one poll: each of 1,500 voters says yes with probability 0.54, and binomial returns the total number of yeses.
print(f'yes answers: {yes} of {n}')  # Print how many said yes out of 1,500.
print(f'sample proportion p-hat = {yes / n:.4f}')  # Divide yeses by n to get the sample proportion p-hat; :.4f shows 4 decimals.

yes answers: 781 of 1500
sample proportion p-hat = 0.5207


**What this means.** This one simulated poll got 781 yes answers out of 1,500, so p-hat = 0.5207. The true support is 0.54, so this poll is off by about 2 points just by luck of the draw. A different poll would give a slightly different answer.

## Step 2: Ten thousand polls

Now run the same poll 10,000 times. Each p-hat differs a little because of sampling variability. Their average should be near p, and their SD should be near the formula.

In [2]:
p_hats = rng.binomial(n, p, size=10_000) / n  # Run the poll 10,000 times at once and turn each yes count into a proportion.

print(f'mean of p-hats          = {p_hats.mean():.4f}')  # Print the average of the 10,000 p-hats; it should sit on the true p.
print(f'SD of p-hats            = {p_hats.std(ddof=1):.4f}')  # Print the SD of the p-hats, the standard error of a poll (ddof=1 is the sample version).
print(f'sqrt(p(1-p)/n) formula  = {np.sqrt(p * (1 - p) / n):.4f}')  # Print the formula square root of p(1-p)/n, which should predict that SD.

mean of p-hats          = 0.5400
SD of p-hats            = 0.0130
sqrt(p(1-p)/n) formula  = 0.0129


**What this means.** The 10,000 polls average 0.5400, exactly the true p, so p-hat has no built-in bias. Their SD is 0.0130 and the formula gives 0.0129, so we can trust the formula. A poll of 1,500 typically lands within about 1.3 points of the truth.

## Step 3: Small samples wobble more

The lesson's three cases. The last column checks the rule that n times p and n times (1 minus p) should both be at least 10 for the normal model to fit well.

In [3]:
print('   n     p   SD of p-hats   formula   both n*p, n*(1-p) >= 10?')  # Print column headings, including the success-failure check.
for n, p in ((20, 0.35), (200, 0.40), (1500, 0.54)):  # Loop over three polls: a tiny one, a medium one and the lesson's big one.
    ph = rng.binomial(n, p, size=10_000) / n  # Simulate 10,000 polls of that size and convert to proportions.
    ok = n * p >= 10 and n * (1 - p) >= 10  # Check the success-failure rule: both n*p and n*(1-p) must be at least 10 for the bell curve to fit.
    print(f'{n:5d}  {p:.2f}   {ph.std(ddof=1):11.4f}   {np.sqrt(p * (1 - p) / n):7.4f}   {ok}')  # Print n, p, the simulated SD, the formula SD and the rule check.

   n     p   SD of p-hats   formula   both n*p, n*(1-p) >= 10?
   20  0.35        0.1053    0.1067   False
  200  0.40        0.0345    0.0346   True
 1500  0.54        0.0129    0.0129   True


**What this means.** The simulated SDs (0.1053, 0.0345, 0.0129) track the formula (0.1067, 0.0346, 0.0129). The small poll of 20 is far noisier, with a spread of about 10 points, and it also fails the success-failure check (False), so the normal model is not trustworthy there.

## Step 4: The margin of error

A 95% margin of error is about 1.96 standard errors. If that is right, about 95% of polls should land within that distance of the true p.

In [4]:
n, p = 1500, 0.54  # Go back to the lesson's poll: 1,500 voters and p = 0.54.
moe = 1.96 * np.sqrt(p * (1 - p) / n)  # Margin of error = 1.96 standard errors, the usual 95% cut-off.
p_hats = rng.binomial(n, p, size=10_000) / n  # Simulate 10,000 polls again.
inside = (np.abs(p_hats - p) <= moe).mean()  # Count the share of polls whose p-hat is within one margin of error of the true p.

print(f'margin of error = {moe:.4f}')  # Print the margin of error.
print(f'share of polls within it = {inside:.3f}')  # Print the share of polls that landed inside it.

margin of error = 0.0252
share of polls within it = 0.951


**What this means.** The margin of error is 0.0252, about 2.5 points, which is where the headline's plus or minus 2.5 comes from. And 0.951 of the simulated polls fell within it, almost exactly the 95% promised.

## So, what is the answer?

Each poll of 1,500 voters lands a little away from the true 0.54, typically about 1.3 points (SD 0.0130, formula 0.0129). That gives a 95% margin of error of 0.0252, the headline's 2.5 points, and 95.1% of simulated polls fell within it. Caution: this margin covers random sampling error only, not bad wording or a biased sample.

## Now you try

A streaming app knows that 35% of its users switch on notifications. A product analyst will survey 300 random users and wants to know how much the sample percentage would vary, and how far off it could be 95% of the time.

**Data:** True proportion p = 0.35, sample size n = 300. Simulate with rng = np.random.default_rng(8) and 10,000 surveys.

1. Check the success-failure condition.
2. Find the standard error from the formula and by simulating 10,000 surveys.
3. Compute the 95% margin of error.
4. What share of simulated surveys land within it?

*Hint:* Success-failure: n*p and n*(1-p) both at least 10. SE = sqrt(p(1-p)/n), margin = 1.96 * SE. Use rng.binomial(n, p, size=10_000) / n for the surveys.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
rng = np.random.default_rng(8)
p, n = 0.35, 300
se = np.sqrt(p * (1 - p) / n)
moe = 1.96 * se
p_hats = rng.binomial(n, p, size=10_000) / n
print(f'n*p = {n * p:.0f}, n*(1-p) = {n * (1 - p):.0f}')
print(f'formula SE = {se:.4f}, simulated SD = {p_hats.std(ddof=1):.4f}')
print(f'margin of error = {moe:.4f}')
print(f'share within it = {(np.abs(p_hats - p) <= moe).mean():.3f}')

n*p = 105, n*(1-p) = 195
formula SE = 0.0275, simulated SD = 0.0277
margin of error = 0.0540
share within it = 0.954


**Answer.** n*p = 105 and n*(1-p) = 195, both above 10, so the bell-curve model is fine. The formula SE is 0.0275 and the simulated SD is 0.0277, which agree. The 95% margin of error is 0.0540, about 5.4 points, and 0.954 of the simulated surveys landed within it. So a 300-user survey is only good to about plus or minus 5 points.

## The same question in R, SPSS and Stata

**R**

```r
set.seed(2024)  # Fix the random seed.
p <- 0.54; n <- 1500  # Set the true support p = 0.54 and the poll size n = 1500.
p_hats <- rbinom(10000, n, p) / n  # Simulate 10,000 polls (rbinom) and divide by n to get proportions.
mean(p_hats)  # Print the average of the p-hats.
sd(p_hats)  # Print their SD, the standard error.
sqrt(p * (1 - p) / n)  # Print the formula sqrt(p(1-p)/n) for comparison.
```

How to read it: mean(p_hats) should print about 0.54, sd(p_hats) about 0.013, and the formula line prints 0.01287. The simulated SD and the formula agree. Python gave 0.5400, 0.0130 and 0.0129.

**SPSS** (Transform > Compute Variable (RV.BINOM), then Analyze > Descriptive Statistics > Descriptives)

```
SET SEED=2024.
INPUT PROGRAM.
LOOP poll = 1 TO 10000.
COMPUTE phat = RV.BINOM(1500, 0.54) / 1500.
END CASE.
END LOOP.
END FILE.
END INPUT PROGRAM.
DESCRIPTIVES VARIABLES=phat /STATISTICS=MEAN STDDEV.
```

How to read it: In the Descriptive Statistics table, the Mean of phat should be near 0.540 and the Std. Deviation near 0.0129, matching the formula SQRT(0.54*0.46/1500) = 0.0129. Differences from Python are just random noise.

**Stata**

```
clear
set seed 2024
set obs 10000
generate phat = rbinomial(1500, 0.54) / 1500
summarize phat
display sqrt(0.54 * 0.46 / 1500)
```

How to read it: In the summarize table, Mean should be about 0.540 and Std. dev. about 0.0129. The display line shows .01287, the formula value. They agree, up to small differences from Python caused by random numbers.